# 04 - Optimiser prototype

Purpose: prototype the single-objective stop-selection problem before implementing the MILP in `app/optimisation/`.

This notebook uses brute force on a tiny synthetic problem so the expected optimum is transparent. Production code should use OR-Tools or PuLP.

In [ ]:
from dataclasses import dataclass
from itertools import combinations


@dataclass(frozen=True)
class Origin:
    origin_id: str
    population: int
    imd_decile: int


@dataclass(frozen=True)
class CandidateStop:
    stop_id: str
    cost_gbp: int
    is_interchange: bool = False


origins = [
    Origin("lsoa_a", 900, 1),
    Origin("lsoa_b", 700, 2),
    Origin("lsoa_c", 500, 7),
    Origin("lsoa_d", 600, 1),
]

candidate_stops = [
    CandidateStop("stop_1", cost_gbp=300_000, is_interchange=True),
    CandidateStop("stop_2", cost_gbp=250_000),
    CandidateStop("stop_3", cost_gbp=200_000),
    CandidateStop("stop_4", cost_gbp=150_000),
]

coverage = {
    "stop_1": {"lsoa_a", "lsoa_b"},
    "stop_2": {"lsoa_b", "lsoa_c"},
    "stop_3": {"lsoa_c", "lsoa_d"},
    "stop_4": {"lsoa_d"},
}

## Equity weights

Start simple: higher weight for lower IMD deciles. This can be changed during sensitivity testing, but the optimiser should receive explicit numeric weights.

In [ ]:
def deprivation_weight(imd_decile: int) -> float:
    if not 1 <= imd_decile <= 10:
        raise ValueError(f"IMD decile must be 1-10, got {imd_decile}")
    return 11 - imd_decile


def solution_score(selected_stop_ids: set[str]) -> float:
    covered_origin_ids = set().union(*(coverage[stop_id] for stop_id in selected_stop_ids)) if selected_stop_ids else set()
    return sum(
        origin.population * deprivation_weight(origin.imd_decile)
        for origin in origins
        if origin.origin_id in covered_origin_ids
    )


solution_score({"stop_1", "stop_3"})

## Brute-force optimum for a tiny fixture

This is not the production solver. It is a transparent fixture for checking the MILP later.

In [ ]:
def brute_force_best_solution(
    stops: list[CandidateStop],
    budget_gbp: int,
    max_stops: int,
    require_interchange: bool = True,
) -> dict[str, object]:
    best: dict[str, object] | None = None
    for stop_count in range(1, max_stops + 1):
        for combo in combinations(stops, stop_count):
            selected_ids = {stop.stop_id for stop in combo}
            total_cost = sum(stop.cost_gbp for stop in combo)
            has_interchange = any(stop.is_interchange for stop in combo)
            if total_cost > budget_gbp:
                continue
            if require_interchange and not has_interchange:
                continue
            score = solution_score(selected_ids)
            candidate = {
                "selected_stop_ids": sorted(selected_ids),
                "total_cost_gbp": total_cost,
                "score": score,
                "has_interchange": has_interchange,
            }
            if best is None or candidate["score"] > best["score"]:
                best = candidate
    if best is None:
        raise ValueError("No feasible solution")
    return best


brute_force_best_solution(candidate_stops, budget_gbp=550_000, max_stops=2)

## MILP extraction notes

Move the production formulation into `app/optimisation/`:

- binary `x_j` for selected candidate stops
- binary `y_i` for covered origins
- maximise `sum(w_i * y_i)`
- enforce coverage only when a selected stop covers an origin
- enforce budget, max stops, and interchange constraints
- return selected stops, covered origins, objective value, cost, and feasibility status